<a href="https://colab.research.google.com/github/alexander-toschev/cv-course/blob/main/Tasks/Task5_Invariant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
!pip install --upgrade gspread pandas google-auth
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from IPython.display import display
import random
# Authenticate and create the PyDrive client.
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# FILL THIS
student_name = "ELON MUSK"
group_id = "11-101"

In [ ]:



# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1VIp1PdTVfR4rWP44YYWzz58hVf-_wLpX4NwXmcG7MNo/edit?usp=sharing"
sh = gc.open_by_url(SPREADSHEET_URL)
worksheet = sh.sheet1
score = 0
# Ensure header row exists
if not worksheet.get_all_values():
    worksheet.append_row(["Student Name", "Group","TaskID", "Score"])




In [ ]:
# MAIN NOTEBOOK GOES HERE
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
task_id = "Task5_Invariant"
score = 0
max_score = 15

Сравнить точность модели с аугментациями и без.

Подвергнуть тестовые изображения зашумлению.

Проверить, какая модель более устойчива к шуму.

Дано: CIFAR-10, две модели (s0 - без аугментаций, s1 - с аугментациями).

In [1]:
#YOUR CODE HERE


import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import torch
import torchvision
import torchvision.transforms as transforms

# --- Трансформации ---
# Без аугментаций
transform_plain = transforms.Compose([
    transforms.ToTensor(),
])

# С аугментациями
transform_augmented = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor(),
])

# --- Загрузка датасетов ---
trainset_plain = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_plain)
trainloader_plain = torch.utils.data.DataLoader(trainset_plain, batch_size=128, shuffle=True, num_workers=2)

trainset_augmented = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_augmented)
trainloader_augmented = torch.utils.data.DataLoader(trainset_augmented, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_plain)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# --- Модель ---
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# --- Функция обучения ---
def train_model(trainloader, epochs=10):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    return model

# --- Обучение ---
model_s0 = train_model(trainloader_plain)
model_s1 = train_model(trainloader_augmented)


100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


KeyboardInterrupt: 

In [ ]:
# 🔧 Тест 1: Проверка что точность s1 не хуже s0 на clean-данных
assert acc_s1_clean >= acc_s0_clean - 2, "Слишком большая потеря на clean"
score +=5

# 🔧 Тест 2: Проверка что точность s1 лучше s0 на noisy-данных
assert acc_s1_noisy > acc_s0_noisy, "Модель с аугментациями должна лучше держать шум"
score +=5

# 🔧 Тест 3: Валидация при деформациях
assert abs(acc_s1_rotated - acc_s1_clean) < 10, "Сильная деградация при повороте"
score +=5

In [ ]:
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
# Save the result to Google Sheets
from datetime import datetime

# Get current date and time
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d %H:%M:%S")
worksheet.append_row([student_name,group_id, task_id, score, timestamp])

print(f"Test completed! {student_name}, your score is {score}/{max_score}.")
